# 15a. Tuning Skenario A (Tanpa Cost-Sensitive Learning)

Notebook ini mencari hiperparameter **Random Forest** dan **XGBoost tanpa cost-sensitive learning** (tanpa `class_weight` dan tanpa `sample_weight`) dengan Random Search yang protokolnya sama persis dengan notebook 07 dan 08:

- data: hanya `train_final.csv`
- skor: Macro F1
- `n_iter=20`, `cv=5`, `random_state=42`
- ruang pencarian: sama dengan notebook 07 dan 08

Hasil ini adalah **Skenario A** pada proposal (hybrid tanpa CSL dengan Random Search sendiri). Hasil notebook 07 dan 08 adalah **Skenario B**.

**Yang ditulis:** hanya folder baru `hasil_eksperimen/15a_tuning_skenario_a`. Folder `artifacts` dan `data` tidak disentuh, dan `test_final_locked.csv` tidak dibaca.

**Waktu:** sama beratnya dengan notebook 07 dan 08 (20 kandidat x 5 lipatan untuk tiap model). Hasil tiap model disimpan begitu selesai, sehingga jika notebook terhenti, model yang sudah selesai tidak diulang.

Mode uji cepat (`UJI_CEPAT = True`) hanya untuk memeriksa bahwa notebook berjalan di komputer Anda (sekitar 1-2 menit). Hasilnya ditulis ke folder terpisah `hasil_eksperimen/15a_uji_cepat` dan **tidak boleh dipakai** sebagai hasil penelitian.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas. Aman dijalankan berulang.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

## 1. Pengaturan

In [ ]:
import json
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import f1_score, make_scorer

from xgboost import XGBClassifier

UJI_CEPAT = False        # True = uji cepat 1-2 menit, hasil BUKAN untuk penelitian

N_ITER = 20
CV = 5
RANDOM_STATE = 42
TRAIN_PATH = "data/final_split/train_final.csv"
OUTPUT_DIR = "hasil_eksperimen/15a_tuning_skenario_a"

if UJI_CEPAT:
    N_ITER = 2
    CV = 2
    OUTPUT_DIR = "hasil_eksperimen/15a_uji_cepat"

os.makedirs(OUTPUT_DIR, exist_ok=True)
JSON_PATH = os.path.join(OUTPUT_DIR, "params_skenario_A.json")

print("Mode uji cepat:", UJI_CEPAT)
print("Folder output :", OUTPUT_DIR)

## 2. Membaca Data Train Final

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)

X_train = df_train.drop(columns=["Diabetes_012"])
y_train = df_train["Diabetes_012"].astype(int)

if UJI_CEPAT:
    X_train, _, y_train, _ = train_test_split(
        X_train, y_train, train_size=6000, stratify=y_train, random_state=0
    )

print("Ukuran data train :", X_train.shape)
print("Distribusi kelas  :")
print(y_train.value_counts().sort_index())
print("\nSkenario A: TIDAK ada class_weight dan TIDAK ada sample_weight.")

## 3. Scorer dan Ruang Pencarian

Ruang pencarian disalin dari notebook 07 (Random Forest) dan 08 (XGBoost) agar kedua skenario dibandingkan secara adil.

In [ ]:
macro_f1_scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)

param_dist_rf = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

param_dist_xgb = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7]
}

# Parameter Skenario B (hasil notebook 07 dan 08) sebagai pembanding
PARAM_B = {
    "rf": {"n_estimators": 400, "min_samples_split": 10, "min_samples_leaf": 2,
           "max_features": "sqrt", "max_depth": 20, "bootstrap": False},
    "xgb": {"n_estimators": 300, "min_child_weight": 5, "max_depth": 4,
            "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 1.0},
}


def ke_json(o):
    if hasattr(o, "item"):
        return o.item()
    return str(o)


if os.path.exists(JSON_PATH):
    with open(JSON_PATH, encoding="utf-8") as f:
        hasil = json.load(f)
    print("File hasil sebelumnya ditemukan. Model yang sudah selesai:", list(hasil.keys()))
else:
    hasil = {}


def simpan_hasil():
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(hasil, f, indent=2, default=ke_json, ensure_ascii=False)

## 4. Random Search Random Forest (Tanpa CSL)

Sel ini memerlukan waktu cukup lama. Jika model ini sudah pernah selesai, sel ini dilewati.

In [ ]:
if "rf" in hasil:
    print("Random Forest skenario A sudah selesai sebelumnya, dilewati.")
else:
    print("=" * 75)
    print("RANDOM SEARCH RANDOM FOREST - SKENARIO A (TANPA CSL)")
    print("=" * 75)

    t0 = time.time()
    rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)   # tanpa class_weight

    search_rf = RandomizedSearchCV(
        estimator=rf_model,
        param_distributions=param_dist_rf,
        n_iter=N_ITER,
        scoring=macro_f1_scorer,
        cv=CV,
        verbose=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    search_rf.fit(X_train, y_train)

    hasil["rf"] = {
        "best_params": search_rf.best_params_,
        "best_macro_f1_cv": round(float(search_rf.best_score_), 4),
        "n_iter": N_ITER,
        "cv": CV,
        "menit": round((time.time() - t0) / 60, 1),
    }
    simpan_hasil()

    cv_rf = pd.DataFrame(search_rf.cv_results_).sort_values("rank_test_score")
    cv_rf[["rank_test_score", "mean_test_score", "std_test_score", "params"]].to_csv(
        os.path.join(OUTPUT_DIR, "cv_results_rf_skenario_A.csv"), index=False)

    print("\nBest Parameters:", search_rf.best_params_)
    print("Best Macro F1 CV:", round(search_rf.best_score_, 4))

## 5. Random Search XGBoost (Tanpa CSL)

In [ ]:
if "xgb" in hasil:
    print("XGBoost skenario A sudah selesai sebelumnya, dilewati.")
else:
    print("=" * 75)
    print("RANDOM SEARCH XGBOOST - SKENARIO A (TANPA CSL)")
    print("=" * 75)

    t0 = time.time()
    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search_xgb = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=param_dist_xgb,
        n_iter=N_ITER,
        scoring=macro_f1_scorer,
        cv=CV,
        verbose=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    search_xgb.fit(X_train, y_train)          # tanpa sample_weight

    hasil["xgb"] = {
        "best_params": search_xgb.best_params_,
        "best_macro_f1_cv": round(float(search_xgb.best_score_), 4),
        "n_iter": N_ITER,
        "cv": CV,
        "menit": round((time.time() - t0) / 60, 1),
    }
    simpan_hasil()

    cv_xgb = pd.DataFrame(search_xgb.cv_results_).sort_values("rank_test_score")
    cv_xgb[["rank_test_score", "mean_test_score", "std_test_score", "params"]].to_csv(
        os.path.join(OUTPUT_DIR, "cv_results_xgb_skenario_A.csv"), index=False)

    print("\nBest Parameters:", search_xgb.best_params_)
    print("Best Macro F1 CV:", round(search_xgb.best_score_, 4))

## 6. Perbandingan Parameter Skenario A dan B

In [ ]:
print("=" * 75)
print("PARAMETER SKENARIO A (TANPA CSL) vs SKENARIO B (DENGAN CSL)")
print("=" * 75)

for nama, kunci in [("Random Forest", "rf"), ("XGBoost", "xgb")]:
    print(f"\n{nama}")
    print("  Skenario A:", hasil[kunci]["best_params"], "| Macro F1 CV =", hasil[kunci]["best_macro_f1_cv"])
    print("  Skenario B:", PARAM_B[kunci])

print("\nCATATAN:")
print("Macro F1 CV skenario A dan B tidak boleh dibandingkan langsung sebagai bukti manfaat CSL,")
print("karena kedua skenario dievaluasi pada lipatan yang dipakai untuk memilih parameternya.")
print("Perbandingan yang adil dilakukan di notebook 15 (satu protokol, mean dan simpangan baku per lipatan).")
print("\nHasil tersimpan di:", JSON_PATH)
print("test_final_locked.csv TIDAK digunakan. Folder artifacts TIDAK diubah.")